# Pulldown Y SNPs with OY SNPs

In [1]:
import numpy as np
import os  # For Saving to Folder
import pandas as pd
import matplotlib.pyplot as plt

import socket
import os as os
import sys as sys
import multiprocessing as mp
from pysam import AlignmentFile

### For Arial Font
from matplotlib import rcParams
rcParams['font.family'] = 'sans-serif'   # Set the defaul
### Make sure to have the font installed (it is on cluster for Harald)
rcParams['font.sans-serif'] = ['Arial']

socket_name = socket.gethostname()
print(socket_name)

if socket_name.startswith("compute-"):
    print("HSM Computational partition detected.")
    path = "/n/groups/reich/hringbauer/git/y_chrom/"  # The Path on Midway Cluster
    
elif socket_name.startswith("bionc") or socket_name.startswith("hpc"):
    print("Leipzig Cluster detected!")
    path = "/mnt/archgen/users/hringbauer/git/y_chrom/"
    
else:
    raise RuntimeWarning("Not compatible machine. Check!!")

os.chdir(path)  # Set the right Path (in line with Atom default)

# Show the current working directory. Should be HAPSBURG/Notebooks/ParallelRuns
print(os.getcwd())
print(f"CPU Count: {mp.cpu_count()}")
print(sys.version)

### Custom Imports
from python.pulldown import load_snp_file_ISOGG, call_y_bam, mismatch_path

hpc030
Leipzig Cluster detected!
/mnt/archgen/users/hringbauer/git/y_chrom
CPU Count: 128
3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]


### Helper Functions

In [2]:
### Prepare Dictionary of Levels

def create_parent_dct(path_parents="/mnt/archgen/users/eric_garcia/OYdb/OYchpar.csv"):
    """"Create Dictionary of Parent Nodes"""
    chpar = {}
    
    # Create a dictionary to store all child-parent relations.
    with open(path_parents, "r") as f:
        for line in f:
            items = line.strip().split(sep=",")
            if not items:
                continue
                    
            child = items[0]
            parent = items[1]
                
            chpar[child] = parent
    
    return chpar


# Create a function to get the total number of SNPs per branch and divide between derived and ancestral.
def div_anc_der(df, chpar=[]):
    """
    Calculate total, ancestral, derived, and uncovered SNPs per branch.
    """
    if len(chpar)==0:
        chpar = create_parent_dct()
    
    # Create columns for ANC and DER SNPs.
    df = df.copy()
    df["ancestral"] = df["alt#"] < df["ref#"]
    df["derived"]   = df["alt#"] > df["ref#"]

    # Divide the df by the different haplogroups.
    grouped = df.groupby("Y-haplogroup")

    # Generate a new df with columns for the Branch, the Level and the total for SNPs.
    new_df = grouped.agg(
        Level=("Level", "first"),
        Total_SNPs=("Y-haplogroup", "size"),
        Ancestral=("ancestral", "sum"),
        Derived=("derived", "sum"),
    ).reset_index()

    # Compute uncovered as the difference of the total with ancestral and derived.
    new_df["Uncovered"] = (
        new_df["Total_SNPs"]
        - new_df["Ancestral"]
        - new_df["Derived"]
    )

    # Rename column called Y-haplogroup
    new_df = new_df.rename(columns={"Y-haplogroup": "Branch"})
    new_df = new_df.sort_values(by="Level")

    # Use previously defined function to look for all ANC and DER SNPs in parental branches. First, transform into a dictionary to speed up processing.
    anc_lookup = new_df.set_index("Branch")["Ancestral"].to_dict()
    der_lookup = new_df.set_index("Branch")["Derived"].to_dict()
    
    anc_par = []
    der_par = []

    # For each branch, find the total number of ancestral and derived SNPs in parent using ancder_par() function.
    for branch in new_df["Branch"]:
        anc_par.append(ancder_par(branch, anc_lookup, chpar=chpar))
        der_par.append(ancder_par(branch, der_lookup, chpar=chpar))

    # Insert new columns.
    new_df["#ANC in par."] = anc_par
    new_df["#DER in par."] = der_par
    
    # Save a csv file
    #new_df.to_csv(f"data/branches_statitstics_{sample}.csv")

    return new_df

def ancder_par(string, par_dict, chpar):
    """ Find the total number of ancestral and derived SNPs for all parental branches"""

    if string not in chpar:
        return 0

    else:
        parent = chpar.get(string)
        return ancder_par(parent, par_dict, chpar) + par_dict.get(parent, 0)

def get_mismatch_snps(string, chpar, df_ch):
    """Get all SNPs that are mismatches of Haplogroup String"""
    dfs_mm = [] # List of mismatching SNP dfs
    
    while True:        
        ### Find all mismatches
        dft = df_ch[df_ch["Y-haplogroup"]==string] # All SNPs in Node
        dfd =dft[dft["ref#"]>dft["alt#"]]
        dfs_mm.append(dfd)  	

        if string not in chpar:
            dfs_mm = pd.concat(dfs_mm)
            return dfs_mm
        
        else:
            string = chpar[string] # Get Parent Node

# 0) Prepare Data

### 0a) Load Autorun Eager Dictionary mapping iids to bam files

In [3]:
dft = pd.read_csv("/mnt/archgen/users/hringbauer/git/auto_popgen/output/TF/v0.3/bam_paths.tsv", sep="\t")
dft2 = pd.read_csv("/mnt/archgen/users/hringbauer/git/auto_popgen/output/RM/v0.3/bam_paths.tsv", sep="\t")
n = np.sum(dft["bam#"]>0)
bam_dict = dict(zip(dft["iid"], dft["bam_path"]))
bam_dict2 = dict(zip(dft2["iid"], dft2["bam_path"]))
print(f"Loaded {len(dft)} Individuals. With BAM: {n}")
print(f"Loaded {len(dft2)} RM Individuals.")

Loaded 28636 Individuals. With BAM: 17215
Loaded 4607 RM Individuals.


### 0b) Prepare SNP List

In [ ]:
### Load ISOGG SNPs
df = load_snp_file_ISOGG("./data/all_snps.csv")

### Load OY SNPs
df1 = pd.read_csv("/mnt/archgen/users/hringbauer/git/y_chrom/data/all_snps_filtered_levels.csv", low_memory=False)
print(f"Loaded {len(df1)} OY SNPs with levels loaded")

### Load OY node dictionary
chpar = create_parent_dct()

### [1x requirement] Create BED file for OY 

In [25]:
savepath = "./data/OY_snps.bed"

dft = df1.sort_values(by="pos")
dft = dft[["chrom", "pos"]].copy()

dft["pos1"] = dft["pos"]
dft.to_csv(savepath, sep="\t", index=False, header=None)
print(f"Saved {len(dft)} OY SNPs to {savepath}")

Saved 2868884 OY SNPs to ./data/OY_snps.bed


### 1) Run Y Calling

### 1a) Single Example with ISOGG SNPs

In [62]:
%%time
path_bam = bam_dict2["KKG002"]

df_ch, df_der = call_y_bam(df=df, 
                           path_bam=path_bam) #A55903 and A55904
len(df_der)

Average Coverage: 5.1489x
#Sites covered: 58158/73148
#Derived Loci: 
1030 / 58158 covered>0
CPU times: user 99.5 ms, sys: 2.7 ms, total: 102 ms
Wall time: 2.57 s


1030

In [63]:
df_der[-100:-50]

### This sample is J2a1a1b1a - looks like 2 SNPs are derived there

,Name,chrom,pos,ref,alt,Subgroup Name,Alternate Names,rs numbers,A,C,G,T,ref#,alt#
930,PF4965,Y,15610713,G,T,J2a,CTS4360,NaN,0,0,0,10,0,10
931,CTS4699,Y,15775488,A,G,J2a,PF4909,NaN,0,0,3,0,0,3
932,PF4966,Y,15704139,C,T,J2a,CTS4540,NaN,0,0,0,5,0,5
933,F841,Y,6854256,C,T,J2a,PF4948,NaN,0,0,0,5,0,5
934,CTS9538,Y,18952077,G,A,J2a,PF4919,NaN,1,0,0,0,0,1
935,L505,Y,21970721,G,T,J2a,PF4987,NaN,0,0,0,1,0,1
936,PF4953,Y,7680253,C,G,J2a,NaN,NaN,0,0,6,0,0,6
937,PF4952,Y,7676739,C,T,J2a,NaN,NaN,0,0,0,7,0,7
938,PF5112,Y,23033706,C,T,J2a1,CTS11251,NaN,0,0,0,1,0,1
939,PF4610,Y,22757708,G,C,J2a1,NaN,NaN,0,3,0,0,0,3


In [61]:
s = "J2a1a1b1a"
mismatch_path(s, df_ch).sort_values(by="Subgroup Name")[-50:]

Mismatches: 0 / 0


,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#


### 1b) Use OY SNPs

In [65]:
%%time
path_bam = bam_dict2["KKG002"]

df_ch, df_der = call_y_bam(df=df1, path_bam=path_bam,
                           path_bed='/mnt/archgen/users/hringbauer/git/y_chrom/data/OY_snps.bed') 
len(df_der)

Average Coverage: 1.3794x
#Sites covered: 907725/2868884
#Derived Loci: 
6981 / 907725 covered>0
CPU times: user 1.28 s, sys: 432 ms, total: 1.71 s
Wall time: 22.7 s


6981

In [ ]:
df_der.sort_values(by="Level")[-50:]

In [67]:
dfc = df_der.groupby("Y-haplogroup").agg(
    n=('Y-haplogroup', 'count'),
    level=('Level', 'mean')).reset_index()
dfc2 = dfc[dfc["n"]>1] # Only extract Y haplogroups that have 2 SNPs derived

In [ ]:
dfc2.sort_values(by="level")[-50:]

In [87]:
df_der[df_der["Y-haplogroup"]=="J-PF5172"]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#


In [85]:
df_der[df_der["Subgroup Name"]=="J-Y153763"]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#


In [81]:
df_der[df_der["Subgroup Name"].str.contains("PF5190")]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#


The Y-Haplogroup seems to be J-Y153763. It is two G->A, but the parent haplogroup, PF5172, also has two G-A derived.

# 2) Run Henry II

In [49]:
%%time
path_bam = "/mnt/archgen/Autorun_eager/eager_outputs/SG/BMG/BMG001/trimmed_bam/BMG001_ss_libmerged_udghalf.trimmed.bam"

df_ch, df_der = call_y_bam(df=df1, path_bam=path_bam,
                           path_bed='/mnt/archgen/users/hringbauer/git/y_chrom/data/OY_snps.bed') 

Average Coverage: 0.3605x
#Sites covered: 813297/2868884
#Derived Loci: 
3469 / 813297 covered>0
CPU times: user 1.24 s, sys: 417 ms, total: 1.65 s
Wall time: 31.3 s


In [ ]:
### Do the parental SNPs

In [50]:
dft = div_anc_der(df_ch)
dfd =dft[dft["Derived"]>dft["Ancestral"]]

In [51]:
dfd[-50:]

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.
29831,O-Y76085,30,1,0,1,0,100,154
28487,O-F14904,31,1,0,1,0,102,154
54511,R-L151,31,1,0,1,0,16,238
21158,J-FTA80624,32,1,0,1,0,67,143
2782,E-FT159421,33,1,0,1,0,153,80
16870,I-Y57591,33,1,0,1,0,81,144
10914,I-FGC13016,33,1,0,1,0,82,144
39615,R-DF88,34,4,0,4,0,16,239
58287,R-YP3980,34,1,0,1,0,44,191
35595,R-BY3293,35,1,0,1,0,23,237


In [73]:
dfd.sort_values(by="#DER in par.").tail()

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.
57669,R-Y6237,38,1,0,1,0,16,244
58652,R-Z17121,43,1,0,1,0,16,245
48157,R-FTA63879,45,2,0,2,0,16,246
48154,R-FTA63331,46,1,0,1,0,16,248
1269,D-CTS3946,9,1,0,1,0,92,250


In [56]:
df_ch[df_ch["Y-haplogroup"]=="R-FTA63331"]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
835808,FTA65919,Y,22139468,T,C,R-FTA63879,NaN,45,0,1,0,0,0,1
838593,FTA63879,Y,14861556,T,A,R-FTA63879,NaN,45,1,0,0,0,0,1


In [67]:
df_mms = get_mismatch_snps("R-FTA63331", chpar=chpar, df_ch=df_ch)

In [68]:
df_mms

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
737640,"FTA9712,Y137",Y,7451254,A,G,R-M269,NaN,27,1,0,0,0,1,0
717305,"FGC66,MF803527",Y,7081561,T,C,R-M343,NaN,22,0,0,0,1,1,0
699628,FGC222,Y,28699018,G,A,IJK-L15,IJK,11,0,0,3,0,3,0
700349,"MF796482,TY198493,V1295",Y,7629583,G,A,IJK-L15,IJK,11,0,0,1,0,1,0
697702,"MF808158,PF1911",Y,23729951,T,C,F-M89,F,8,0,0,0,1,1,0
697918,"L508,TY200867",Y,22755855,G,A,F-M89,F,8,0,0,1,0,1,0
698069,"FGC2048,MF801858",Y,28459738,G,A,F-M89,F,8,0,0,2,0,2,0
698176,"CTS9317,MF806850,PF1767",Y,18818812,T,C,F-M89,F,8,0,0,0,1,1,0
695514,Z9315,Y,16933354,A,C,A-V168,NaN,3,2,0,0,0,2,0
695616,"Y9964,Z9440",Y,18132698,T,C,A-V168,NaN,3,0,0,0,1,1,0


In [ ]:
127395 	MF784814,YSC0000166 	Y 	14116584 	A 	T 	R-P297 	NaN 	26 	17 	0 	0 	0 	17 	0
2090274 	FGC58,MF803532 	Y 	7100362 	T 	C 	R-M343 	NaN 	22 	0 	0 	0 	8 	8 	0
2090443 	FGC66,MF803527 	Y 	7081561 	T 	C 	R-M343 	NaN 	22 	0 	0 	0 	9 	9 	0
2073713 	FGC280,MF786133 	Y 	19298321 	A 	G 	R-UTY2 	NaN 	20 	9 	0 	0 	0 	9 	0
2075838 	MF789201,YSC0000067 	Y 	7133986 	C 	G 	R-UTY2 	NaN 	20 	0 	8 	0 	0 	8 	0
2037704 	FGC222 	Y 	28699018 	G 	A 	IJK-L15 	IJK 	11 	0 	0 	9 	0 	9 	0
2039873 	MF796482,TY198493,V1295 	Y 	7629583 	G 	A 	IJK-L15 	IJK 	11 	0 	0 	9 	0 	9 	0
2030716 	FGC2646 	Y 	14565310 	A 	C 	F-M89 	F 	8 	10 	0 	0 	0 	10 	0
2031171 	CTS5750 	Y 	16467111 	T 	C 	F-M89 	F 	8 	0 	0 	0 	6 	6 	0
2032012 	MF808158,PF1911 	Y 	23729951 	T 	C 	F-M89 	F 	8 	0 	0 	0 	7 	7 	0
2032053 	MF806436,PF1720,TY199699 	Y 	17142068 	T 	A 	F-M89 	F 	8 	0 	0 	0 	9 	9 	0
2032660 	L508,TY200867 	Y 	22755855 	G 	A 	F-M89 	F 	8 	0 	0 	8 	0 	8 	0
2033114 	FGC2048,MF801858 	Y 	28459738 	G 	A 	F-M89 	F 	8 	0 	0 	5 	0 	5 	0
2033423 	CTS9317,MF806850,PF1767 	Y 	18818812 	T 	C 	F-M89 	F 	8 	0 	0 	0 	10 	10 	0
2028214 	Y1487 	Y 	5693239 	A 	G 	CT-M168 	NaN 	6 	3 	0 	1 	0 	3 	1
2028770 	Y1445 	Y 	2987520 	C 	T 	CT-M168 	NaN 	6 	0 	9 	0 	5 	9 	5
2025570 	Z9315 	Y 	16933354 	A 	C 	A-V168 	NaN 	3 	19 	0 	0 	0 	19 	0
2025623 	V6478,Z9327 	Y 	17028360 	T 	A 	A-V168 	NaN 	3 	0 	0 	0 	8 	8 	0
2025917 	Y9964,Z9440 	Y 	18132698 	T 	C 	A-V168 	NaN 	3 	0 	0 	0 	7 	7 	0
2025928 	Z9298 	Y 	16816310 	A 	G 	A-V168 	NaN 	3 	5 	0 	0 	0 	5 	0
2026307 	CTS12370 	Y 	28583229 	T 	C 	A-V168 	NaN 	3 	0 	0 	0 	10 	10 	0
2024913 	FGC24654 	Y 	10012683 	C 	T 	A-L1090 	NaN 	2 	0 	3 	0 	0 	3 	0
2024924 	A5111 	Y 	13511050 	C 	G 	A-L1090 	NaN 	2 	0 	10 	9 	0 	10 	9
2024950 	FGC27794 	Y 	23049484 	T 	C 	A-L1090 	NaN 	2 	0 	0 	0 	9 	9 	0
2024961 	BY184462 	Y 	8055446 	G 	A 	A-L1090 	NaN 	2 	0 	0 	16 	0 	16 	0
2024981 	FGC27824 	Y 	23139472 	T 	C 	A-L1090 	NaN 	2 	0 	0 	0 	13 	13 	0
2025136 	V1615 	Y 	8098483 	T 	A 	A-L1090 	NaN 	2 	0 	0 	0 	11 	11 	0
2025140 	FGC26134 	Y 	13832165 	G 	T 	A-L1090 	NaN 	2 	0 	0 	8 	0 	8 	0
2025150 	V3167 	Y 	15833573 	T 	C 	A-L1090 	NaN 	2 	0 	0 	0 	7 	7 	0

## 2b) High-Coverage PTN sample

In [74]:
%%time
path_bam = "/mnt/archgen/Autorun_eager/eager_outputs/SG/PTN/PTN209/trimmed_bam/PTN209_ss_libmerged_udghalf.trimmed.bam"

df_ch, df_der = call_y_bam(df=df1, path_bam=path_bam,
                           path_bed='/mnt/archgen/users/hringbauer/git/y_chrom/data/OY_snps.bed') 

Average Coverage: 7.2976x
#Sites covered: 2370738/2868884
#Derived Loci: 
6390 / 2370738 covered>0
CPU times: user 2.54 s, sys: 1.74 s, total: 4.27 s
Wall time: 51.7 s


In [ ]:
df_der.sort_values(by="Level")[-50:]

In [88]:
dft = div_anc_der(df_ch)
dfd =dft[dft["Derived"]>dft["Ancestral"]]
dfd.sort_values(by="#DER in par.").tail(20)

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.
84668,R-M269,27,97,0,97,0,29,589
1448,C-V9,7,38,3,35,0,286,656
84564,R-L23,28,3,0,3,0,29,686
84587,R-L51,29,5,0,5,0,29,689
55893,R-BY3293,35,1,0,1,0,43,689
84552,R-L151,31,3,0,3,0,29,694
84890,R-PF6538,31,1,0,1,0,29,694
50798,R-BY1188,53,1,0,1,0,52,697
84558,R-L2,36,1,0,1,0,29,697
56679,R-BY3953,45,1,0,1,0,39,697


In [83]:
df_ch[df_ch["Y-haplogroup"]=="R-FTA63879"]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
2437353,FTA65919,Y,22139468,T,C,R-FTA63879,NaN,45,0,0,0,7,7,0
2441200,"FTA65732,PR6621",Y,21593639,T,C,R-FTA63879,NaN,45,0,0,0,13,13,0
2445506,FTA63879,Y,14861556,T,A,R-FTA63879,NaN,45,0,0,0,11,11,0


In [84]:
df_mms = get_mismatch_snps("R-FTG53091", chpar=chpar, df_ch=df_ch)

In [87]:
df_mms[:]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
2127395,"MF784814,YSC0000166",Y,14116584,A,T,R-P297,NaN,26,17,0,0,0,17,0
2090274,"FGC58,MF803532",Y,7100362,T,C,R-M343,NaN,22,0,0,0,8,8,0
2090443,"FGC66,MF803527",Y,7081561,T,C,R-M343,NaN,22,0,0,0,9,9,0
2073713,"FGC280,MF786133",Y,19298321,A,G,R-UTY2,NaN,20,9,0,0,0,9,0
2075838,"MF789201,YSC0000067",Y,7133986,C,G,R-UTY2,NaN,20,0,8,0,0,8,0
2037704,FGC222,Y,28699018,G,A,IJK-L15,IJK,11,0,0,9,0,9,0
2039873,"MF796482,TY198493,V1295",Y,7629583,G,A,IJK-L15,IJK,11,0,0,9,0,9,0
2030716,FGC2646,Y,14565310,A,C,F-M89,F,8,10,0,0,0,10,0
2031171,CTS5750,Y,16467111,T,C,F-M89,F,8,0,0,0,6,6,0
2032012,"MF808158,PF1911",Y,23729951,T,C,F-M89,F,8,0,0,0,7,7,0


# Other St. Pölten males 1x WGS

In [7]:
%%time
path_bam = "/mnt/archgen/Autorun_eager/eager_outputs/SG/PTN/PTN267/trimmed_bam/PTN267_ss.A0101_udghalf.trimmed.bam"

df_ch, df_der = call_y_bam(df=df1, path_bam=path_bam,
                           path_bed='/mnt/archgen/users/hringbauer/git/y_chrom/data/OY_snps.bed') 

Average Coverage: 0.4835x
#Sites covered: 1004319/2868884
#Derived Loci: 
4110 / 1004319 covered>0
CPU times: user 1.42 s, sys: 549 ms, total: 1.96 s
Wall time: 26.1 s


In [8]:
dft = div_anc_der(df_ch)
dfd =dft[dft["Derived"]>dft["Ancestral"]]
dfd.sort_values(by="#DER in par.").tail(20)

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.
23932,J-FTD43091,36,1,0,1,0,67,197
26001,J-Y3082,33,1,0,1,0,55,197
16557,I-M170,14,46,1,45,0,14,197
25985,J-Y304060,34,1,0,1,0,105,197
22844,J-FTA47014,29,1,0,1,0,72,197
24460,J-FTF47360,30,1,0,1,0,31,197
16877,I-S31,15,14,0,14,0,15,242
10286,I-BY167444,24,1,0,1,0,33,242
11657,I-CTS2257,16,10,0,10,0,15,256
16878,I-S33,18,21,1,20,0,15,266


In [11]:
df_mms = get_mismatch_snps("I-FT58623", chpar=chpar, df_ch=df_ch)
df_mms

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
876403,"L37,PF6900,S153",Y,17516123,T,C,I-S33,NaN,18,0,0,0,1,1,0
869930,Y1934,Y,8504226,G,A,I-M170,I,14,0,0,1,0,1,0
867820,PF3518,Y,6607318,C,T,IJ-P124,NaN,12,0,2,0,0,2,0
868010,PF3561,Y,21389837,G,A,IJ-P124,NaN,12,0,0,1,0,1,0
866516,FGC222,Y,28699018,G,A,IJK-L15,IJK,11,0,0,1,0,1,0
867413,"MF796482,TY198493,V1295",Y,7629583,G,A,IJK-L15,IJK,11,0,0,3,0,3,0
863674,FGC2646,Y,14565310,A,C,F-M89,F,8,2,0,0,0,2,0
864223,"MF808158,PF1911",Y,23729951,T,C,F-M89,F,8,0,0,0,1,1,0
864239,"MF806436,PF1720,TY199699",Y,17142068,T,A,F-M89,F,8,0,0,0,2,2,0
861566,"V6478,Z9327",Y,17028360,T,A,A-V168,NaN,3,0,0,0,1,1,0


### Son of above sample

In [15]:
%%time
path_bam = "/mnt/archgen/Autorun_eager/eager_outputs/SG/PTN/PTN139/trimmed_bam/PTN139_ss_libmerged_udghalf.trimmed.bam"

df_ch, df_der = call_y_bam(df=df1, path_bam=path_bam,
                           path_bed='/mnt/archgen/users/hringbauer/git/y_chrom/data/OY_snps.bed') 

Average Coverage: 0.3297x
#Sites covered: 752016/2868884
#Derived Loci: 
3882 / 752016 covered>0
CPU times: user 1.24 s, sys: 493 ms, total: 1.73 s
Wall time: 19.5 s


In [16]:
dft = div_anc_der(df_ch)
dfd =dft[dft["Derived"]>dft["Ancestral"]]
dfd.sort_values(by="#DER in par.").tail(20)

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.
48105,R-FTC3117,55,1,0,1,0,94,147
11214,I-FT235980,32,1,0,1,0,18,185
14769,I-S31,15,8,0,8,0,10,185
10477,I-FGC88432,23,1,0,1,0,18,185
15658,I-Y36690,33,1,0,1,0,19,185
10389,I-FGC52744,46,1,0,1,0,21,185
10212,I-CTS2257,16,4,0,4,0,10,193
14770,I-S33,18,14,1,13,0,10,197
9415,I-BY3095,29,1,0,1,0,40,210
14864,I-Y10720,20,2,0,2,0,11,210


In [17]:
df_mms = get_mismatch_snps("I-FT58623", chpar=chpar, df_ch=df_ch)
df_mms

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
655590,BY31307,Y,13690483,G,T,I-S33,NaN,18,0,0,1,0,1,0
648726,"MF796482,TY198493,V1295",Y,7629583,G,A,IJK-L15,IJK,11,0,0,2,0,2,0
646057,CTS5750,Y,16467111,T,C,F-M89,F,8,0,0,0,1,1,0
646332,"MF806436,PF1720,TY199699",Y,17142068,T,A,F-M89,F,8,0,0,0,1,1,0
646758,"CTS9317,MF806850,PF1767",Y,18818812,T,C,F-M89,F,8,0,0,0,5,5,0
644359,Z9315,Y,16933354,A,C,A-V168,NaN,3,1,0,0,0,1,0
644181,FGC27824,Y,23139472,T,C,A-L1090,NaN,2,0,0,0,1,1,0
644233,V1615,Y,8098483,T,A,A-L1090,NaN,2,0,0,0,1,1,0
644235,FGC26134,Y,13832165,G,T,A-L1090,NaN,2,0,0,2,0,2,0
644239,V3167,Y,15833573,T,C,A-L1090,NaN,2,0,0,0,1,1,0


In [ ]:
PTN146